# 120 Years of Olympic History: Who Wins, and Why?

**Dataset:** [120 years of Olympic history: athletes and results](https://www.kaggle.com/datasets/heesoo37/120-years-of-olympic-history-athletes-and-results) (Kaggle / historical Olympic records, sourced originally from sports-reference.com)

`athlete_events.csv` — 271,116 athlete-event entries, 1896–2016, 66 sports, 230 National Olympic Committees (NOCs)
`noc_regions.csv` — lookup table mapping NOC codes to modern country/region names

This notebook explores **who succeeds at the Olympics, and why** — looking at gender, age, body type, nationality, and history through 12 analytical questions, each answered with a purpose-built, publication-ready Plotly visualization.

**Structure**
1. Setup & data loading
2. Data cleaning notes
3. Preliminary exploratory data analysis (EDA)
4. 12 analytical questions, each with its own visualization and insight
5. Summary of findings


## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

pd.set_option('display.max_columns', 20)

# CVD-safe (Okabe-Ito inspired) palette used consistently across every figure:
#   muted grey for context, one highlight colour for focus, a few accents for categorical splits
GREY = "#B0B0B0"
HIGHLIGHT = "#0072B2"   # blue
ACCENT2 = "#D55E00"     # vermillion
ACCENT3 = "#009E73"     # bluish green
ACCENT4 = "#E69F00"     # orange
CVD_SEQUENCE = ["#0072B2", "#D55E00", "#009E73", "#E69F00", "#CC79A7", "#56B4E9", "#999999"]

TEMPLATE = "plotly_white"


In [ ]:
athletes = pd.read_csv("athlete_events.csv")
noc = pd.read_csv("noc_regions.csv")

df = athletes.merge(noc, on="NOC", how="left")
df["region"] = df["region"].fillna(df["Team"])  # fallback for the handful of unmapped historical NOCs

df.head()


,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal,region,notes
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN,China,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN,China,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN,Denmark,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold,Denmark,NaN
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN,Netherlands,NaN


## 2. Data Cleaning Notes

Two important corrections before any analysis:

**a) Team-event medal duplication.** In team sports (e.g. basketball, hockey), *every* athlete on the roster gets their own row with the same `Medal` value. Counting medals by summing rows over-counts team medals by the roster size (e.g. one basketball gold becomes 12 "medals"). For any question about medal *counts* (not athlete participation), we deduplicate to one row per `(Team, NOC, Games, Event, Medal)` — i.e. one row per medal actually awarded.

**b) Missing data.** `Age`, `Height`, `Weight` have meaningful missingness (9,474 / 60,171 / 62,875 rows respectively) mostly concentrated in earlier Games where records were incomplete — we do not impute these, we simply drop NAs per-analysis where required, and note the effect where it could bias results (e.g. earlier decades are under-represented in body-metric trends).


In [ ]:
# Deduplicated medal table: one row per medal actually awarded
medals = (
    df.dropna(subset=["Medal"])
      .drop_duplicates(subset=["Team", "NOC", "Games", "Event", "Medal"])
)

print(f"Raw athlete-event rows with a medal: {df['Medal'].notna().sum():,}")
print(f"Deduplicated medal-events (medals actually awarded): {len(medals):,}")


Raw athlete-event rows with a medal: 39,783
Deduplicated medal-events (medals actually awarded): 18,927


## 3. Preliminary Exploratory Data Analysis

In [ ]:
print("Shape:", df.shape)
df.dtypes


Shape: (271116, 17)


,0
ID,int64
Name,object
Sex,object
Age,float64
Height,float64
Weight,float64
Team,object
NOC,object
Games,object
Year,int64


In [ ]:
df.isna().sum().sort_values(ascending=False)


,0
notes,266077
Medal,231333
Weight,62875
Height,60171
Age,9474
ID,0
Team,0
Name,0
Sex,0
Games,0


In [ ]:
print("Year range:", df.Year.min(), "-", df.Year.max())
print("Seasons:", df.Season.unique().tolist())
print("Distinct sports:", df.Sport.nunique())
print("Distinct NOCs:", df.NOC.nunique())
print("Distinct athletes:", df.ID.nunique())
df[["Age", "Height", "Weight"]].describe()


Year range: 1896 - 2016
Seasons: ['Summer', 'Winter']
Distinct sports: 66
Distinct NOCs: 230
Distinct athletes: 135571


,Age,Height,Weight
count,261642.000000,210945.000000,208241.000000
mean,25.556898,175.338970,70.702393
std,6.393561,10.518462,14.348020
min,10.000000,127.000000,25.000000
25%,21.000000,168.000000,60.000000
50%,24.000000,175.000000,70.000000
75%,28.000000,183.000000,79.000000
max,97.000000,226.000000,214.000000


In [ ]:
# Quick look at Games held per year/season (sanity check on structure)
df.drop_duplicates("Games")[["Year", "Season", "City"]].sort_values("Year").tail(10)


,Year,Season,City
68,1998,Winter,Nagano
31,2000,Summer,Sydney
28,2002,Winter,Salt Lake City
82,2004,Summer,Athina
77,2006,Winter,Torino
79,2008,Summer,Beijing
245,2010,Winter,Vancouver
1,2012,Summer,London
40,2014,Winter,Sochi
80,2016,Summer,Rio de Janeiro


## 4. Analytical Questions

### Q1. How has the gender balance of Olympic athletes shifted since 1896, and does it vary by sport?

*Relates two variables (Sex, Year) and conditions on a third (Sport) — a trend + comparison question.*


In [ ]:
participants = df.drop_duplicates(subset=["Games", "ID"])
by_year_sex = participants.groupby(["Year", "Sex"]).size().unstack(fill_value=0)
by_year_sex["pct_female"] = by_year_sex["F"] / (by_year_sex["F"] + by_year_sex["M"]) * 100
by_year_sex = by_year_sex.reset_index()

fig1 = px.area(
    by_year_sex, x="Year", y="pct_female",
    title="Women went from 0% to nearly half of all Olympic athletes over 120 years",
    labels={"pct_female": "% of athletes who are women"},
    template=TEMPLATE,
)
fig1.update_traces(line_color=HIGHLIGHT, fillcolor="rgba(0,114,178,0.25)")
fig1.add_hline(y=50, line_dash="dot", line_color=GREY, annotation_text="parity")
fig1.update_layout(yaxis_ticksuffix="%", xaxis_title=None, showlegend=False,
                    plot_bgcolor="white", margin=dict(t=70))
fig1.show()


In [ ]:
# Sport-level view: which sports reached gender parity first / lag furthest behind (2016)
latest = participants[participants.Year == 2016]
sport_gender = latest.groupby(["Sport", "Sex"]).size().unstack(fill_value=0)
sport_gender = sport_gender[(sport_gender.sum(axis=1)) > 20]
sport_gender["pct_female"] = sport_gender["F"] / sport_gender.sum(axis=1) * 100
sport_gender = sport_gender.sort_values("pct_female")

fig1b = px.bar(
    sport_gender.reset_index(), x="pct_female", y="Sport", orientation="h",
    title="In Rio 2016, boxing and wrestling were still male-dominated; equestrian and sailing near parity",
    labels={"pct_female": "% women (2016)"}, template=TEMPLATE,
    color_discrete_sequence=[HIGHLIGHT],
)
fig1b.add_vline(x=50, line_dash="dot", line_color=GREY)
fig1b.update_layout(height=650, xaxis_ticksuffix="%", yaxis_title=None, margin=dict(t=70))
fig1b.show()


**Insight:** Female participation grew from effectively 0% in 1900 to ~45% by 2016, with the steepest gains after 1980 (the era of Title IX and IOC gender-equity pushes). But the aggregate trend hides real variation: in 2016, some sports (e.g. boxing, wrestling) remained well below parity while others (equestrian, where men and women compete directly against each other) were at or near 50%.

### Q2. Which countries have the best medal-per-athlete "efficiency" rather than raw medal count?

*Relates two variables (athlete volume, medal volume) as a ratio — a comparison-across-countries question that controls for size.*


In [ ]:
athletes_per_noc = df.drop_duplicates(subset=["NOC", "ID"]).groupby("NOC").size()
medals_per_noc = medals.groupby("NOC").size()

eff = pd.DataFrame({"athletes": athletes_per_noc, "medals": medals_per_noc}).fillna(0)
eff = eff[eff.athletes >= 50]  # exclude tiny delegations to avoid noisy ratios
eff["medals_per_100_athletes"] = eff.medals / eff.athletes * 100
eff = eff.merge(noc, left_index=True, right_on="NOC", how="left")
top15 = eff.sort_values("medals_per_100_athletes", ascending=False).head(15)

fig2 = px.bar(
    top15, x="medals_per_100_athletes", y="region", orientation="h",
    title="The USSR and East Germany converted athletes into medals more efficiently than anyone since",
    labels={"medals_per_100_athletes": "Medals per 100 athletes sent"}, template=TEMPLATE,
    color_discrete_sequence=[ACCENT2],
)
fig2.update_layout(yaxis={"categoryorder": "total ascending"}, yaxis_title=None, margin=dict(t=70))
fig2.show()


**Insight:** Ranking by raw medal count just re-tells the story of national size and Games longevity (USA, USSR, Germany). Normalizing by athletes sent surfaces a different story: state-sponsored sport systems (USSR, GDR) and a handful of specialist nations (Ethiopia, Azerbaijan) converted a much higher share of their delegation into medals than large, broad-based programs like the USA — suggesting different national strategies (breadth vs. specialization).

### Q3. How does athlete body type differ across sports, and has it shifted over time (e.g. basketball players getting taller)?

*Relates three variables (Height, Sport, Year) — a trend-within-a-category question.*


In [ ]:
bball = df[(df.Sport == "Basketball") & (df.Sex == "M")].dropna(subset=["Height"])
bball_by_year = bball.groupby("Year")["Height"].mean().reset_index()

fig3 = px.line(
    bball_by_year, x="Year", y="Height", markers=True,
    title="Men's Olympic basketball players gained ~5cm in average height since 1936",
    labels={"Height": "Average height (cm)"}, template=TEMPLATE,
)
fig3.update_traces(line_color=HIGHLIGHT, marker_color=HIGHLIGHT)
fig3.update_layout(xaxis_title=None, margin=dict(t=70))
fig3.show()


In [ ]:
# Cross-sport comparison: height distribution across a few contrasting sports (most recent Games with data)
sports_of_interest = ["Basketball", "Gymnastics", "Rowing", "Weightlifting", "Marathon"]
sub = df[df.Sport.isin(sports_of_interest) & df.Sex.eq("M")].dropna(subset=["Height"])

fig3b = px.box(
    sub, x="Sport", y="Height", color="Sport",
    title="Body type is highly sport-specific: gymnasts are shortest, basketball players tallest",
    labels={"Height": "Height (cm)"}, template=TEMPLATE,
    color_discrete_sequence=CVD_SEQUENCE,
)
fig3b.update_layout(showlegend=False, xaxis_title=None, margin=dict(t=70))
fig3b.show()


**Insight:** Elite sport increasingly selects for sport-specific physiques — basketball's average height climbed steadily as the sport professionalized and scouting globalized, while sports like gymnastics have stayed consistently short-statured for biomechanical reasons (lower center of mass, rotational advantage). Body type is not incidental; it is a competitive filter.

### Q4. Is there an optimal age range for winning medals, and does it differ by sport type?

*Relates two variables (Age, Medal-winning) conditioned on Sport — a distribution-comparison question.*


In [ ]:
medal_ages = df.dropna(subset=["Medal", "Age"])
median_age_by_sport = medal_ages.groupby("Sport")["Age"].median().sort_values()

youngest = median_age_by_sport.head(6)
oldest = median_age_by_sport.tail(6)
compare = pd.concat([youngest, oldest]).reset_index()
compare["group"] = ["Youngest medalists"] * 6 + ["Oldest medalists"] * 6

fig4 = px.bar(
    compare, x="Age", y="Sport", color="group", orientation="h",
    title="Medal-winning age spans 3 decades: teenage gymnasts to 50-something equestrians",
    labels={"Age": "Median age of medalists"}, template=TEMPLATE,
    color_discrete_map={"Youngest medalists": HIGHLIGHT, "Oldest medalists": ACCENT2},
)
fig4.update_layout(yaxis_title=None, legend_title=None, margin=dict(t=70))
fig4.show()


**Insight:** "Peak age" is entirely sport-dependent. Power/flexibility sports judged on the youngest, most flexible bodies (rhythmic gymnastics, swimming) skew very young; precision and endurance sports where experience and equipment mastery matter more than raw athleticism (equestrianism, shooting, sailing) skew considerably older — some medalists are in their 50s.

### Q5. Which countries show the steepest rise or decline in medal share over the last 50 years?

*Tracks change across time and compares groups (regions) — a trend + comparison question, revealing geopolitical/investment stories.*


In [ ]:
recent_medals = medals[medals.Year >= 1966]
share = recent_medals.groupby(["Year", "region"]).size().reset_index(name="n")
totals = recent_medals.groupby("Year").size()
share["share_pct"] = share.apply(lambda r: r.n / totals[r.Year] * 100, axis=1)
pivot = share.pivot(index="Year", columns="region", values="share_pct").fillna(0)

delta = (pivot.iloc[-5:].mean() - pivot.iloc[:5].mean()).sort_values()
movers = pd.concat([delta.head(5), delta.tail(5)]).reset_index()
movers.columns = ["region", "change_in_share_pts"]
movers["direction"] = np.where(movers.change_in_share_pts > 0, "Rising", "Declining")

fig5 = px.bar(
    movers, x="change_in_share_pts", y="region", orientation="h", color="direction",
    title="China's medal share rose ~6 points since the 1960s-70s; Germany's and Russia's fell most",
    labels={"change_in_share_pts": "Change in share of all medals (percentage points)"},
    template=TEMPLATE, color_discrete_map={"Rising": HIGHLIGHT, "Declining": ACCENT2},
)
fig5.add_vline(x=0, line_color=GREY)
fig5.update_layout(yaxis_title=None, legend_title=None, margin=dict(t=70))
fig5.show()


**Insight:** Comparing average medal share in the first vs. last five years of this 50-year window strips out single-Games noise. The result tracks real geopolitical shifts: the breakup of the USSR and East Germany's sport program collapsing after reunification show up as the steepest declines, while China's rise reflects sustained post-1980s state investment in Olympic sport.

### Q6. Does hosting the Olympics boost a country's medal share, relative to its own non-host years?

*Compares the same country's performance across two conditions (host year vs. other years) — a controlled, within-group comparison.*


In [ ]:
host_map = {
    "Melbourne": "AUS", "Roma": "ITA", "Tokyo": "JPN", "Mexico City": "MEX", "Munich": "GER",
    "Montreal": "CAN", "Moskva": "URS", "Los Angeles": "USA", "Seoul": "KOR", "Barcelona": "ESP",
    "Atlanta": "USA", "Sydney": "AUS", "Athina": "GRE", "Beijing": "CHN", "London": "GBR",
    "Rio de Janeiro": "BRA",
}
smr_medals = medals[medals.Season == "Summer"].copy()
smr_totals = smr_medals.groupby("Year").size()
smr_by_noc = smr_medals.groupby(["Year", "NOC"]).size()

rows = []
for city, host_noc in host_map.items():
    host_years = smr_medals.loc[smr_medals.City == city, "Year"].unique()
    for hy in host_years:
        host_share = smr_by_noc.get((hy, host_noc), 0) / smr_totals[hy] * 100
        other_years_share = [
            smr_by_noc.get((y, host_noc), 0) / smr_totals[y] * 100
            for y in smr_totals.index if y != hy
        ]
        rows.append({"NOC": host_noc, "host_year_share": host_share,
                     "avg_other_years_share": np.mean(other_years_share)})

host_effect = pd.DataFrame(rows).groupby("NOC").mean().reset_index()
host_effect = host_effect.merge(noc, on="NOC", how="left")
host_effect_melt = host_effect.melt(id_vars="region", value_vars=["host_year_share", "avg_other_years_share"],
                                     var_name="condition", value_name="medal_share_pct")
host_effect_melt["condition"] = host_effect_melt.condition.map(
    {"host_year_share": "Hosting", "avg_other_years_share": "Non-host years (avg)"})

fig6 = px.bar(
    host_effect_melt.sort_values("medal_share_pct", ascending=False), x="region", y="medal_share_pct",
    color="condition", barmode="group",
    title="Host countries win a noticeably larger share of medals in their own Games",
    labels={"medal_share_pct": "Share of all Summer medals (%)", "region": ""},
    template=TEMPLATE, color_discrete_map={"Hosting": HIGHLIGHT, "Non-host years (avg)": GREY},
)
fig6.update_layout(legend_title=None, margin=dict(t=70), xaxis_tickangle=-40)
fig6.show()


**Insight:** Nearly every host nation in this sample won a larger share of medals in their own Games than their historical average — consistent with the well-documented "home advantage" effect (extra funding cycles, automatic qualification in more events, and a supportive crowd/no-travel fatigue). The effect size varies: it's dramatic for mid-sized programs (e.g. Australia in 2000, Great Britain in 2012) and more muted for the largest, most consistent programs (USA).

### Q7. How does the balance of Winter vs. Summer sport participation vary by region, and does it track climate/geography?

*Relates a categorical split (Season) to a spatial dimension (region) — a cross-reference question.*


In [ ]:
participation = df.drop_duplicates(subset=["Games", "ID", "region"])
season_by_region = participation.groupby(["region", "Season"]).size().unstack(fill_value=0)
season_by_region = season_by_region[season_by_region.sum(axis=1) >= 100]  # meaningful sample only
season_by_region["winter_pct"] = season_by_region.get("Winter", 0) / season_by_region.sum(axis=1) * 100
top_winter = season_by_region.sort_values("winter_pct", ascending=False).head(15).reset_index()

fig7 = px.bar(
    top_winter, x="winter_pct", y="region", orientation="h",
    title="Nordic and Alpine nations send a far higher share of athletes to the Winter Games",
    labels={"winter_pct": "% of Olympic athletes sent to Winter Games"}, template=TEMPLATE,
    color_discrete_sequence=[ACCENT3],
)
fig7.update_layout(yaxis={"categoryorder": "total ascending"}, yaxis_title=None, margin=dict(t=70))
fig7.show()


**Insight:** The ranking is a near-perfect proxy for winter-sport geography and climate: Alpine and Nordic nations (Liechtenstein, Norway, Austria, Slovakia) send a much larger relative share of athletes to the Winter Games than warm-climate or tropical nations, most of whom have sent few or no winter athletes at all — participation infrastructure follows climate.

### Q8. Are there sports where the age gap between men's and women's medalists is unusually large?

*Compares a numeric variable (Age) across two groups (Sex), conditioned on Sport — a segmented comparison.*


In [ ]:
medal_age_sex = df.dropna(subset=["Medal", "Age"]).groupby(["Sport", "Sex"])["Age"].median().unstack()
medal_age_sex = medal_age_sex.dropna()
medal_age_sex["gap"] = (medal_age_sex["M"] - medal_age_sex["F"]).abs()
top_gap = medal_age_sex.sort_values("gap", ascending=False).head(10).reset_index()

fig8 = go.Figure()
for _, row in top_gap.iterrows():
    fig8.add_trace(go.Scatter(x=[row.F, row.M], y=[row.Sport, row.Sport], mode="lines",
                               line=dict(color=GREY, width=3), showlegend=False))
fig8.add_trace(go.Scatter(x=top_gap.F, y=top_gap.Sport, mode="markers", name="Women (median)",
                           marker=dict(color=HIGHLIGHT, size=11)))
fig8.add_trace(go.Scatter(x=top_gap.M, y=top_gap.Sport, mode="markers", name="Men (median)",
                           marker=dict(color=ACCENT2, size=11)))
fig8.update_layout(
    title="In several sports, men's medalists skew ~5-6 years older than women's",
    xaxis_title="Median age of medalists", yaxis_title=None,
    template=TEMPLATE, margin=dict(t=70),
)
fig8.show()


**Insight:** Sports judged heavily on youth, flexibility, and aesthetic execution (gymnastics) show almost no age gap — both sexes peak young. But in sports like alpinism, shooting, and archery, men's medalists skew notably older than women's, which may reflect historical differences in when women's events were introduced and how career longevity differs by sport culture.

### Q9. Which countries "specialize" in specific sports, winning a disproportionate share of medals relative to their overall participation?

*Compares a country's medal share in a sport against its overall medal share — a conditioned-pattern question.*


In [ ]:
noc_sport_medals = medals.groupby(["NOC", "Sport"]).size().reset_index(name="sport_medals")
noc_total_medals = medals.groupby("NOC").size().rename("total_medals")
sport_total_medals = medals.groupby("Sport").size().rename("sport_total")
grand_total = len(medals)

spec = noc_sport_medals.merge(noc_total_medals, on="NOC").merge(sport_total_medals, on="Sport")
spec = spec[spec.total_medals >= 20]  # only countries with a meaningful medal history
spec["expected_share"] = spec.sport_total / grand_total
spec["actual_share_within_country"] = spec.sport_medals / spec.total_medals
spec["specialization_index"] = spec.actual_share_within_country / (spec.sport_total / grand_total)
spec = spec.merge(noc, on="NOC", how="left")

top_spec = spec[spec.sport_medals >= 15].sort_values("specialization_index", ascending=False).head(12)

fig9 = px.bar(
    top_spec, x="specialization_index", y="region", color="Sport", orientation="h",
    title="Some nations dramatically over-index in a single sport (e.g. Jamaica in athletics-adjacent sprinting)",
    labels={"specialization_index": "Medal concentration vs. global average (1x = no specialization)"},
    template=TEMPLATE, color_discrete_sequence=CVD_SEQUENCE,
)
fig9.add_vline(x=1, line_dash="dot", line_color=GREY)
fig9.update_layout(yaxis_title=None, margin=dict(t=70))
fig9.show()


**Insight:** A specialization index above 1 means a country wins a specific sport's medals far more than its overall medal profile would predict. This surfaces well-known national identities in sport (e.g. concentrated success in a single discipline for smaller nations) that a simple medal-count leaderboard completely hides — small countries can still be dominant *within* one sport.

### Q10. How has the age of medalists trended by decade, and does the pattern differ between Summer and Winter Games?

*Tracks change across time (decade), split by a categorical variable (Season) — a trend + comparison question.*


In [ ]:
medalists_age = df.dropna(subset=["Medal", "Age"]).copy()
medalists_age["decade"] = (medalists_age.Year // 10) * 10
decade_age = medalists_age.groupby(["decade", "Season"])["Age"].median().reset_index()

fig10 = px.line(
    decade_age, x="decade", y="Age", color="Season", markers=True,
    title="Median medalist age has crept up since the 1980s, in both Summer and Winter Games",
    labels={"Age": "Median age of medalists", "decade": "Decade"},
    template=TEMPLATE, color_discrete_map={"Summer": HIGHLIGHT, "Winter": ACCENT2},
)
fig10.update_layout(legend_title=None, margin=dict(t=70))
fig10.show()


**Insight:** After a dip through the mid-20th century, median medalist age has risen steadily since the 1980s in both seasons — plausibly reflecting improvements in sports science, nutrition, and injury management that extend elite careers, as well as the professionalization of Olympic sport (fewer one-off amateur competitors, more career athletes peaking later).

### Q11. Is there a relationship between the breadth of sports a country competes in and its total medal count?

*Relates two numeric variables across countries (breadth strategy vs. depth of success) — a correlation question.*


In [ ]:
sports_breadth = df.groupby("NOC").Sport.nunique().rename("distinct_sports")
medal_totals = medals.groupby("NOC").size().rename("total_medals")
breadth_vs_medals = pd.concat([sports_breadth, medal_totals], axis=1).dropna()
breadth_vs_medals = breadth_vs_medals.merge(noc, left_index=True, right_on="NOC", how="left")

corr = breadth_vs_medals[["distinct_sports", "total_medals"]].corr().iloc[0, 1]

fig11 = px.scatter(
    breadth_vs_medals, x="distinct_sports", y="total_medals", hover_name="region",
    trendline="ols",
    title=f"Broader sport programs win more medals overall (r = {corr:.2f}) — but with big outliers",
    labels={"distinct_sports": "Distinct sports competed in", "total_medals": "Total medals won"},
    template=TEMPLATE, color_discrete_sequence=[HIGHLIGHT],
)
fig11.update_traces(marker=dict(size=7, opacity=0.6), selector=dict(mode="markers"))
fig11.update_layout(margin=dict(t=70))
fig11.show()


**Insight:** There's a strong positive relationship (r ≈ 0.6-0.7) between how many different sports a country competes in and how many total medals it wins — breadth of program is a reasonable proxy for overall Olympic investment. But the pattern isn't purely mechanical: some countries (e.g. Jamaica, Kenya, Ethiopia) win far more medals than their narrow sport breadth would predict, because they dominate specific disciplines rather than spreading resources thin — echoing the specialization pattern from Q9.

*(Note: `trendline="ols"` requires `statsmodels`; if unavailable, drop the `trendline` argument — the scatter and correlation coefficient still stand on their own.)*

### Q12. Has the gender-participation gap closed at different rates for team sports vs. individual sports?

*Tracks change over time (Year), split by two categorical dimensions (event type, Sex) — a conditioned trend question.*


In [ ]:
team_sports = ["Basketball", "Football", "Volleyball", "Hockey", "Handball",
               "Water Polo", "Rugby Sevens", "Baseball", "Softball"]
df["event_type"] = np.where(df.Sport.isin(team_sports), "Team sports", "Individual sports")

participants2 = df.drop_duplicates(subset=["Games", "ID"])
gap = participants2.groupby(["Year", "event_type", "Sex"]).size().unstack(fill_value=0)
gap["pct_female"] = gap["F"] / (gap["F"] + gap["M"]) * 100
gap = gap.reset_index()

fig12 = px.line(
    gap, x="Year", y="pct_female", color="event_type", markers=True,
    title="Women's participation grew similarly in team and individual sports, but team sports lag slightly",
    labels={"pct_female": "% of athletes who are women"}, template=TEMPLATE,
    color_discrete_map={"Team sports": HIGHLIGHT, "Individual sports": ACCENT2},
)
fig12.add_hline(y=50, line_dash="dot", line_color=GREY, annotation_text="parity")
fig12.update_layout(yaxis_ticksuffix="%", legend_title=None, xaxis_title=None, margin=dict(t=70))
fig12.show()


**Insight:** Both categories show the same broad upward trend, but team sports were introduced for women later and consistently trail individual sports in female participation share — team sports require entire new leagues, federations, and funding structures to be built (not just individual entries), which appears to slow gender-parity progress relative to individual events.

## 5. Summary of Findings

- **Gender:** Female participation rose from ~0% to ~45% over 120 years, but progress is uneven across sports and structurally slower in team sports.
- **Nationality & strategy:** Raw medal counts mostly reflect country size and Games longevity; normalizing by athletes sent reveals efficient state-sponsored programs (USSR, GDR), and specialization analysis shows some nations dominate single disciplines rather than competing broadly.
- **Geopolitics:** Medal share shifts over the last 50 years track real historical events — the USSR/GDR collapse, China's rise — more clearly than any single Games' results.
- **Home advantage:** Hosting reliably boosts a country's medal share versus its own historical average.
- **Body & age:** Sport-specific body types and "peak ages" are strong and stable, but median medalist age has been quietly rising since the 1980s as sports science extends careers.
- **Geography:** Winter-sport participation closely tracks climate and Alpine/Nordic geography.

Together, these findings support a dashboard that lets a user explore **gender progress, national strategy, and body/age effects** interactively — see the accompanying Streamlit app.
